# BirdCLEF 2026 Notebook
Training and evaluation built inline (soundscape validation subset).

In [5]:

import random
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
import torchaudio.functional as AF
from sklearn.metrics import roc_auc_score
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class SpectrogramTransform(nn.Module):
    def __init__(
        self,
        sample_rate: int = 32000,
        n_mels: int = 128,
        n_fft: int = 2048,
        hop_length: int = 512,
        f_min: int = 20,
        f_max: Optional[int] = None,
        normalize: bool = True,
    ):
        super().__init__()
        self.melspec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=f_min,
            f_max=f_max or sample_rate // 2,
            power=2.0,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
        self.normalize = normalize

    def forward(self, waveform: Tensor) -> Tensor:
        spec = self.melspec(waveform)
        spec = self.to_db(spec)
        if self.normalize:
            mean = spec.mean(dim=[-2, -1], keepdim=True)
            std = spec.std(dim=[-2, -1], keepdim=True).clamp(min=1e-6)
            spec = (spec - mean) / std
        return spec


def _split_label_string(raw_value: Optional[str]) -> List[str]:
    if not raw_value or not isinstance(raw_value, str):
        return []
    return [label for label in re.split(r"[;@,\s]+", raw_value.strip()) if label]


def _time_to_seconds(value: Union[str, float, int]) -> float:
    if isinstance(value, (int, float)):
        return float(value)
    parts = [float(part) for part in value.split(":")]
    seconds = 0.0
    for part in parts:
        seconds = seconds * 60 + part
    return seconds


def _read_audio(filepath: Path) -> Tuple[Tensor, int]:
    data, sr = sf.read(str(filepath), dtype="float32")
    if data.ndim == 1:
        waveform = torch.from_numpy(data).unsqueeze(0)
    else:
        waveform = torch.from_numpy(data.T.copy())
    return waveform, sr


def build_label_map(metadata: pd.DataFrame) -> Dict[str, int]:
    if "primary_label" not in metadata.columns:
        raise ValueError("metadata must contain primary_label")
    labels: List[str] = []
    for raw in metadata["primary_label"].dropna().astype(str):
        for label in _split_label_string(raw):
            if label not in labels:
                labels.append(label)
    if not labels:
        raise ValueError("No primary labels found in metadata")
    return {label: idx for idx, label in enumerate(labels)}


class BirdCLEFDataset(Dataset):
    def __init__(
        self,
        audio_dir: Path,
        metadata: pd.DataFrame,
        label_map: Dict[str, int],
        sample_rate: int = 32000,
        duration: float = 5.0,
        transform: Optional[SpectrogramTransform] = None,
        training: bool = True,
        preload_audio: bool = True,
        evaluation_segments: bool = False,
        return_filename: bool = False,
        fallback_audio_dirs: Optional[List[Path]] = None,
    ):
        self.audio_dir = Path(audio_dir)
        self.audio_dirs = [self.audio_dir]
        if fallback_audio_dirs:
            self.audio_dirs.extend(Path(p) for p in fallback_audio_dirs)
        self.metadata = metadata.reset_index(drop=True)
        self.label_map = label_map
        self.sample_rate = sample_rate
        self.segment_samples = int(duration * sample_rate)
        self.training = training
        self.transform = transform or SpectrogramTransform(sample_rate=sample_rate)
        self.preload_audio = preload_audio
        self.evaluation_segments = evaluation_segments
        self.return_filename = return_filename
        self.audio_cache: Dict[str, Tensor] = {}
        self.row_audio_info: Dict[int, Dict[str, Union[str, int]]] = {}
        self.eval_segments: List[Dict[str, int]] = []
        if self.preload_audio:
            self._preload_audio()
        if self.evaluation_segments:
            self._build_eval_segments()

    def __len__(self) -> int:
        return len(self.eval_segments) if self.evaluation_segments else len(self.metadata)

    def __getitem__(self, idx: int):
        if self.evaluation_segments:
            row_idx = self.eval_segments[idx]["row_idx"]
            row = self.metadata.iloc[row_idx]
            waveform = self._get_cached_waveform(row_idx)
            chunk = self._extract_segment(waveform, row)
            spec = self.transform(chunk)
            target = self._encode_target(row)
            if self.return_filename:
                filename = str(row.get("filename") or row.get("recording_id"))
                return spec, target, filename
            return spec, target
        row = self.metadata.iloc[idx]
        waveform = self._get_cached_waveform(idx)
        chunk = self._extract_segment(waveform, row)
        spec = self.transform(chunk)
        target = self._encode_target(row)
        return spec, target

    def _resolve_audio_path(self, row: pd.Series) -> Path:
        filename = str(row.get("filename") or row.get("recording_id"))
        species_folder = row.get("primary_label") or row.get("common_name")
        species_folder = str(species_folder) if pd.notna(species_folder) else ""
        for base_dir in self.audio_dirs:
            candidate = base_dir / filename
            if candidate.exists():
                return candidate
            if species_folder:
                candidate = base_dir / species_folder / filename
                if candidate.exists():
                    return candidate
        raise FileNotFoundError(f"Missing audio file {filename}")

    def _preload_audio(self) -> None:
        for idx, row in self.metadata.iterrows():
            filepath = self._resolve_audio_path(row)
            key = str(filepath)
            if key not in self.audio_cache:
                waveform, sr = _read_audio(filepath)
                if waveform.size(0) > 1:
                    waveform = waveform.mean(dim=0, keepdim=True)
                if sr != self.sample_rate:
                    waveform = AF.resample(waveform, orig_freq=sr, new_freq=self.sample_rate)
                self.audio_cache[key] = waveform
            total = self.audio_cache[key].size(1)
            self.row_audio_info[idx] = {"filepath": key, "total_samples": total}

    def _get_cached_waveform(self, row_idx: int) -> Tensor:
        info = self.row_audio_info.get(row_idx)
        if not info:
            row = self.metadata.iloc[row_idx]
            filepath = self._resolve_audio_path(row)
            waveform, sr = _read_audio(filepath)
            if waveform.size(0) > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            if sr != self.sample_rate:
                waveform = AF.resample(waveform, orig_freq=sr, new_freq=self.sample_rate)
            key = str(filepath)
            self.audio_cache[key] = waveform
            total = waveform.size(1)
            info = {"filepath": key, "total_samples": total}
            self.row_audio_info[row_idx] = info
        return self.audio_cache[info["filepath"]]

    def _build_eval_segments(self) -> None:
        self.eval_segments = []
        for idx, row in self.metadata.iterrows():
            start_time = row.get("start")
            end_time = row.get("end")
            if pd.isna(start_time) or pd.isna(end_time):
                continue
            self.eval_segments.append({"row_idx": idx})

    def _extract_segment(
        self,
        waveform: Tensor,
        row: pd.Series,
        start_sample: Optional[int] = None,
    ) -> Tensor:
        if start_sample is not None:
            return self._ensure_segment_length(waveform[:, start_sample : start_sample + self.segment_samples])
        start_time = row.get("start")
        end_time = row.get("end")
        if pd.notna(start_time) and pd.notna(end_time):
            return self._slice_by_times(waveform, start_time, end_time)
        num_samples = waveform.size(1)
        if num_samples <= self.segment_samples:
            return self._ensure_segment_length(waveform)
        max_start = num_samples - self.segment_samples
        start_idx = random.randint(0, max_start) if self.training else max_start // 2
        return waveform[:, start_idx : start_idx + self.segment_samples]

    def _slice_by_times(self, waveform: Tensor, start_time: Union[str, float, int], end_time: Union[str, float, int]) -> Tensor:
        total = waveform.size(1)
        start_sample = min(total, max(0, int(_time_to_seconds(start_time) * self.sample_rate)))
        end_sample = min(
            total,
            max(start_sample + 1, int(_time_to_seconds(end_time) * self.sample_rate)),
        )
        chunk = waveform[:, start_sample:end_sample]
        return self._ensure_segment_length(chunk)

    def _ensure_segment_length(self, chunk: Tensor) -> Tensor:
        length = chunk.size(1)
        if length == self.segment_samples:
            return chunk
        if length < self.segment_samples:
            pad_len = self.segment_samples - length
            return F.pad(chunk, (0, pad_len))
        return chunk[:, : self.segment_samples]

    def _encode_target(self, row: pd.Series) -> Tensor:
        labels = _split_label_string(str(row.get("primary_label", "")))
        return self._encode_labels(labels)

    def _encode_labels(self, labels: List[str]) -> Tensor:
        target = torch.zeros(len(self.label_map), dtype=torch.float32)
        for label in labels:
            idx = self.label_map.get(label)
            if idx is not None:
                target[idx] = 1.0
        return target


class SimpleCNN(nn.Module):
    def __init__(self, n_classes: int, in_channels: int = 1, base_channels: int = 64, dropout: float = 0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            self._conv_block(in_channels, base_channels),
            self._conv_block(base_channels, base_channels * 2),
            self._conv_block(base_channels * 2, base_channels * 4),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Flatten(),
            nn.Linear(base_channels * 4, n_classes),
        )

    def _conv_block(self, in_channels: int, out_channels: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.encoder(x)
        x = self.pool(x)
        return self.classifier(x)


def eval_collate_fn(batch):
    specs, targets, filenames = zip(*batch)
    return torch.stack(specs), torch.stack(targets), list(filenames)


def evaluate_model(model: nn.Module, dataloader: DataLoader, criterion, device: torch.device):
    model.eval()
    losses: List[float] = []
    all_targets: List[Tensor] = []
    all_preds: List[Tensor] = []
    with torch.no_grad():
        for batch in dataloader:
            inputs, targets, *_ = batch
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits = model(inputs)
            loss = criterion(logits, targets)
            losses.append(loss.item())
            all_preds.append(torch.sigmoid(logits).cpu())
            all_targets.append(targets.cpu())
    avg_loss = np.mean(losses) if losses else 0.0
    if all_targets:
        target_tensor = torch.cat(all_targets).numpy()
        pred_tensor = torch.cat(all_preds).numpy()
        try:
            auc = roc_auc_score(target_tensor, pred_tensor, average="macro")
        except ValueError:
            auc = float("nan")
    else:
        auc = float("nan")
    return avg_loss, auc


In [6]:

set_seed(42)
data_root = Path('../data')
train_csv = data_root / 'train.csv'
soundscape_csv = data_root / 'train_soundscapes_labels.csv'
train_audio_dir = data_root / 'train_audio'
soundscape_dir = data_root / 'train_soundscapes'

train_df = pd.read_csv(train_csv)
soundscape_df = pd.read_csv(soundscape_csv)
train_df['start'] = pd.NA
train_df['end'] = pd.NA

combined = pd.concat([train_df, soundscape_df], axis=0, ignore_index=True)
label_map = build_label_map(combined)

val_files = sorted(soundscape_df['filename'].unique())[: min(20, len(soundscape_df['filename'].unique()))]
val_df = combined[combined['filename'].isin(val_files)].reset_index(drop=True)
train_df = combined[~combined['filename'].isin(val_files)].reset_index(drop=True)

transform = SpectrogramTransform(sample_rate=32000, n_mels=128, hop_length=512)
train_ds = BirdCLEFDataset(
    audio_dir=train_audio_dir,
    metadata=train_df,
    label_map=label_map,
    transform=transform,
    duration=5.0,
    training=True,
    preload_audio=False,
    fallback_audio_dirs=[soundscape_dir],
)
val_ds = BirdCLEFDataset(
    audio_dir=soundscape_dir,
    metadata=val_df,
    label_map=label_map,
    transform=transform,
    duration=5.0,
    training=False,
    preload_audio=True,
    evaluation_segments=True,
    return_filename=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=eval_collate_fn,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN(n_classes=len(label_map)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = torch.nn.BCEWithLogitsLoss()
best_auc = float('-inf')


In [7]:

from tqdm import tqdm
for epoch in range(1, 4):
    model.train()
    running_loss = 0.0
    for inputs, targets in tqdm(train_loader, desc=f'Epoch {epoch}'):
        inputs = inputs.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    val_loss, val_auc = evaluate_model(model, val_loader, criterion, device)
    print(f'Epoch {epoch}: train_loss={avg_loss:.4f}, val_loss={val_loss:.4f}, val_auc={val_auc:.4f}')
    if val_auc > best_auc:
        best_auc = val_auc
print(f'Best soundscape ROC-AUC over validation files: {best_auc:.4f}')


Epoch 1:   1%|          | 6/1146 [00:11<35:10,  1.85s/it]


KeyboardInterrupt: 

In [ ]:
train_df

,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection,start,end
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364.0,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat,NaN,NaN
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364.0,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat,NaN,NaN
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364.0,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat,NaN,NaN
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364.0,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat,NaN,NaN
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364.0,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36642,555146;65380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BC2026_Train_0055_S22_20220219_200000.ogg,NaN,00:00:35,00:00:40
36643,517063;555146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BC2026_Train_0055_S22_20220219_200000.ogg,NaN,00:00:40,00:00:45
36644,517063;555146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BC2026_Train_0055_S22_20220219_200000.ogg,NaN,00:00:45,00:00:50
36645,517063;555146;66971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BC2026_Train_0055_S22_20220219_200000.ogg,NaN,00:00:50,00:00:55
